Lấy tất cả bài viết trong ngày

In [138]:
import requests
from bs4 import BeautifulSoup
from datetime import date
import time
import random
import json
from langchain.text_splitter import RecursiveCharacterTextSplitter
from datetime import datetime, timedelta

# 1. Cấu hình

In [147]:

headers = {"User-Agent": "Mozilla/5.0"}
ngay = (date.today() - timedelta(days=0)).strftime("%Y-%m-%d")
all_links = []
page = 1

# 2. Lấy danh sách link

In [140]:
print(f"=== LINK NGÀY {ngay} ===")
while True:
    if page == 1:
        url = f"https://dantri.com.vn/thoi-su/from/{ngay}/to/{ngay}.htm"
    else:
        url = f"https://dantri.com.vn/thoi-su/from/{ngay}/to/{ngay}/trang-{page}.htm"

    res = requests.get(url, headers=headers, timeout=10)
    soup = BeautifulSoup(res.text, "html.parser")

    articles = soup.find_all(
        "article",
        class_="article-item",
        attrs={"data-content-name": f"category-timeline_page_{page}"}
    )

    if not articles:
        break

    for art in articles:
        link = art.get("data-content-target")
        if link:
            if link.startswith("/"):
                link = "https://dantri.com.vn" + link
                all_links.append(link)
    page += 1
print("\nTỔNG LINK:", len(all_links))
for l in all_links:
    print(l)


=== LINK NGÀY 2026-02-06 ===

TỔNG LINK: 52
https://dantri.com.vn/thoi-su/lai-o-to-vuot-au-trong-thoi-tiet-suong-mu-nam-tai-xe-bi-phat-5-trieu-dong-20260206231555094.htm
https://dantri.com.vn/thoi-su/khong-khi-lanh-manh-tran-ve-mien-bac-tu-trua-chieu-nay-20260206210931994.htm
https://dantri.com.vn/thoi-su/tong-bi-thu-ket-thuc-tot-dep-chuyen-tham-cap-nha-nuoc-toi-vuong-quoc-campuchia-20260206224700629.htm
https://dantri.com.vn/thoi-su/tuyen-bo-chung-giua-viet-nam-va-campuchia-20260206224143584.htm
https://dantri.com.vn/thoi-su/cau-long-bien-gap-su-co-nganh-duong-sat-phong-toa-khan-cap-de-sua-chua-20260206214932186.htm
https://dantri.com.vn/thoi-su/viet-nam-se-dao-tao-boi-duong-kien-thuc-cong-tac-dan-toc-cho-can-bo-campuchia-20260206214709054.htm
https://dantri.com.vn/thoi-su/tong-bi-thu-to-lam-gap-can-bo-dai-su-quan-va-cong-dong-nguoi-viet-o-campuchia-20260206213544101.htm
https://dantri.com.vn/thoi-su/bo-chinh-tri-ra-quy-dinh-ve-muc-dong-dang-phi-20260206210920568.htm
https://dantri.co

# 3. Hàm crawl

In [141]:
def crawl_article(url):
    res = requests.get(url, headers=headers, timeout=10)
    soup = BeautifulSoup(res.text, "html.parser")

    # ===== TITLE =====
    h1 = soup.find("h1")
    title = h1.get_text(strip=True) if h1 else ""

    # ===== DATE =====
    # Tìm thẻ <time> có thuộc tính datetime, bất kể class là gì
    time_tag = soup.find("time", attrs={"datetime": True})
    if time_tag:
        # Ưu tiên lấy giá trị trong thuộc tính datetime (thường chuẩn hóa hơn)
        published_date = time_tag.get("datetime")
        # Nếu muốn lấy text hiển thị (ví dụ: "Thứ bảy, 24/01/2026 - 11:45") thì dùng:
        # published_date = time_tag.get_text(strip=True)
    else:
        published_date = ""

    # ===== BODY =====
    paragraphs = []
    for p in soup.find_all("p"):
        text = p.get_text(strip=True)
        if text:
            paragraphs.append(text)

    body = "\n".join(paragraphs)

    return {
        "url": url,
        "title": title,
        "date": published_date,
        "content": body
    }

In [142]:
print("\n=== TEST 1 BÀI ===")
if all_links:
    test_url = all_links[0] # Lấy bài đầu tiên để test
    data = crawl_article(test_url)

    print("URL:", data["url"])
    print("TITLE:", data["title"])
    print("DATE:", data["date"])
    print("CONTENT:")
    print(data["content"][:500] + "...") # In 500 ký tự đầu
else:
    print("Không tìm thấy link nào để test.")



=== TEST 1 BÀI ===
URL: https://dantri.com.vn/thoi-su/lai-o-to-vuot-au-trong-thoi-tiet-suong-mu-nam-tai-xe-bi-phat-5-trieu-dong-20260206231555094.htm
TITLE: Lái ô tô vượt ẩu trong thời tiết sương mù, nam tài xế bị phạt 5 triệu đồng
DATE: 2026-02-07 00:00
CONTENT:
Tối 6/2, Cục CSGT (Bộ Công an) cho biết, Phòng CSGT Công an tỉnh Lào Cai vừa lập biên bản xử phạt với tài xế B.Đ.Q. (SN 1991, quê Ninh Bình) do có hành vi vượt ẩu trên đường.
Với vi phạm trên, tài xế Q. bị phạt 5 triệu đồng, trừ 2 điểm giấy phép lái xe.
Hình ảnh tài xế Q. điều khiển ô tô vượt ẩu trên đường, trong thời tiết sương mù (Ảnh: Cắt từ clip).
Theo Cục CSGT, vào lúc 16h50 ngày 3/2, tại km53+500 tỉnh lộ 155 thuộc địa phận phường Sa Pa, tỉnh Lào Cai, tài xế Q. điều khiển ô tô biển số 18A-2...


# 4. Cấu hình Splitter

In [143]:
if 'data' in locals() and data:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=800,
        chunk_overlap=150,
        separators=[
            "\n\n",   # đoạn
            "\n",     # dòng
            ".", "!", "?", ":", ";", ","
        ]
    )
    dulieu = splitter.split_text(data["content"])
    print("Số chunk:", len(dulieu))
    for i, chunk in enumerate(dulieu):
        print(f"\n--- CHUNK {i} ---")
        print(chunk)


Số chunk: 2

--- CHUNK 0 ---
Tối 6/2, Cục CSGT (Bộ Công an) cho biết, Phòng CSGT Công an tỉnh Lào Cai vừa lập biên bản xử phạt với tài xế B.Đ.Q. (SN 1991, quê Ninh Bình) do có hành vi vượt ẩu trên đường.
Với vi phạm trên, tài xế Q. bị phạt 5 triệu đồng, trừ 2 điểm giấy phép lái xe.
Hình ảnh tài xế Q. điều khiển ô tô vượt ẩu trên đường, trong thời tiết sương mù (Ảnh: Cắt từ clip).
Theo Cục CSGT, vào lúc 16h50 ngày 3/2, tại km53+500 tỉnh lộ 155 thuộc địa phận phường Sa Pa, tỉnh Lào Cai, tài xế Q. điều khiển ô tô biển số 18A-254.xx rồi có hành vi lấn làn, vượt ẩu trên đường. Việc làm của tài xế Q. suýt gây ra vụ tai nạn giao thông.

--- CHUNK 1 ---
Đáng nói, hành vi vi phạm của tài xế Q. bị camera hành trình của một ô tô đi chiều ngược lại ghi hình, người dân đã phản ánh vụ việc tới Thiếu tướng Đỗ Thanh Bình, Cục trưởng Cục CSGT. Sau khi nhận tin báo, Phòng CSGT Công an Lào Cai vào cuộc làm rõ.
Theo cảnh sát, qua xác minh, lực lượng chức năng xác định tài xế B.Đ.Q. đã có hành vi vượt xe g

# 5. Xử lý hàng loạt

In [144]:
final_data = []
print("\n=== BẮT ĐẦU XỬ LÝ CHI TIẾT BÀI VIẾT ===")

for i, link in enumerate(all_links):
    print(f"[{i+1}/{len(all_links)}] Đang xử lý: {link}")

    article_data = crawl_article(link)

    if article_data and article_data["content"]:
        # Split text
        chunks = splitter.split_text(article_data["content"])

        # Lưu kết quả
        final_data.append({
            "url": link,
            "title": article_data["title"],
            "date": article_data["date"],
            "chunks": chunks
        })
        print(f"  -> OK: {len(chunks)} chunks | Date: {article_data['date']}")
    else:
        print("  -> Bỏ qua (không lấy được nội dung)")

    # Rate limiting
    sleep_time = random.uniform(1, 2)
    time.sleep(sleep_time)

print(f"\n=== HOÀN THÀNH ===")
print(f"Đã xử lý thành công: {len(final_data)} bài.")



=== BẮT ĐẦU XỬ LÝ CHI TIẾT BÀI VIẾT ===
[1/52] Đang xử lý: https://dantri.com.vn/thoi-su/lai-o-to-vuot-au-trong-thoi-tiet-suong-mu-nam-tai-xe-bi-phat-5-trieu-dong-20260206231555094.htm
  -> OK: 2 chunks | Date: 2026-02-07 00:00
[2/52] Đang xử lý: https://dantri.com.vn/thoi-su/khong-khi-lanh-manh-tran-ve-mien-bac-tu-trua-chieu-nay-20260206210931994.htm
  -> OK: 4 chunks | Date: 2026-02-07 00:00
[3/52] Đang xử lý: https://dantri.com.vn/thoi-su/tong-bi-thu-ket-thuc-tot-dep-chuyen-tham-cap-nha-nuoc-toi-vuong-quoc-campuchia-20260206224700629.htm
  -> OK: 10 chunks | Date: 2026-02-06 22:47
[4/52] Đang xử lý: https://dantri.com.vn/thoi-su/tuyen-bo-chung-giua-viet-nam-va-campuchia-20260206224143584.htm
  -> OK: 28 chunks | Date: 2026-02-06 22:41
[5/52] Đang xử lý: https://dantri.com.vn/thoi-su/cau-long-bien-gap-su-co-nganh-duong-sat-phong-toa-khan-cap-de-sua-chua-20260206214932186.htm
  -> OK: 6 chunks | Date: 2026-02-06 22:04
[6/52] Đang xử lý: https://dantri.com.vn/thoi-su/viet-nam-se-dao-t

In [145]:
# 6. Lưu file JSON
output_file = "dulieu.json"
result = {
    "date": ngay,
    "articles": final_data
} 
with open(output_file, "w", encoding="utf-8") as f:
    json.dump(result, f, ensure_ascii=False, indent=4)
print(f"Đã lưu dữ liệu vào file: {output_file}")

Đã lưu dữ liệu vào file: dulieu.json
